### Lesson 1.1 - Tokenization and Preprocessing
---

<b>1.1.1: Why Tokenization?</b><br>
Computer doesn't understand text data. It can only understand numbers and therefore works on numeric data. When we want to train a neural network by feeding a text, first WE need to convert it to numbers. This process of conversion is called as <b>Tokenization</b>

<b>Example:</b><br>
```python
Input: "I love machine learning"
Output: [0, 5, 12, 8]  (integer IDs)
```

<b>1.1.2: The 3 levels of Tokenization</b><br>

<b>1.1.1: Why Tokenization?</b><br>

<b>Level 1: Character-level Tokenization</b><br>
* Every character is considered as a single/distinct token
* Example: `"hello"` -> `['h', 'e', 'l', 'l', 'o']`
* Pros: Vocabulary is very small/managable (26 alphabets + special characters)
* Cons: Sequences are very large, "hello" word comprises of 5 tokens alone. The patterns cannot be learnt by the model, not a convinient method.

<b>Level 2: Word-level Tokenization</b><br>
* Every word is considered as a single token
* Example: `"I love cats"` -> `['I', 'love', 'cats']`
* Pros: Meaningful units. Sequences are smaller.
* Cons: 
    - OOV Problem - If the model is trained on "cat" and tested on "cats" then it will get confused. (Both the words should be present in the training set)
    <br>
    - Vocabulary size: Every word from a standard dictionary needs to be fed to the model. For example, our `vocab_size` is 50,000 then the Embedding layer becomes `50,000 x embedding_dim`. This is very large for a model to handle.

<b>Level 3: Subword Tokenization (the modern standard)</b><br>
* Words are broken  down into smaller, logical chunks
* Example: `"lower"` -> `['low', 'er]` or `['lo', 'wer']`
* Pros: 
    - Rare words (like "lower") are broken down into common subwords
    - Common words (like "the") remains a single token
    - OOV problem is solved
    - Vocabulary size is much more manageable
* Cons: It is having a relatively a complex implementation process
* Used by: GPT, BERT, LLaMA, T5 (by Google) - Almost all modern LLMs

<b>This notebook emphasizes on Subword Tokenization, specifically BPE (Byte Pair Encoding)</b>

<b>1.1.3: What is BPE?</b><br>
BPE is not a neural network, it is a data compression algorithm that was adapted in NLP. It has a simple core idea:

<b>Intuition</b>
* In the beginning, every character is a separate token
* Then we merge the most frequently co-occuring character pairs
* Example: If 'l' and 'o' are together most of the time then they are merged into 'lo'
* This process runs iteratively until the vocabulary reaches a specified `target_size`

<b>Real-World example:</b><br>

```js
Word: "lower"
Initial: ['l', 'o', 'w', 'e', 'r']
After some merges: ['lo', 'w', 'er']
```

Note: Now 'lo' and 'er' are two different tokens that are together most of the times. If the model encounters the word "lover" our tokenizer can handle it as only "v" is different.

### Math Behind it:
---

<b>1.2.1: BPE Algorithm Step-by-step</b>
Input: A corpus (list of sentences), target vocab_size

Step 1: Initialize Base Vocabulary

* Every unique characters are appended into our vocabulary
* Example: If corpus has "low", "lower", the vocab becomes = {'l', 'o', 'w', 'e', 'r'}

Step 2: Represent Corpus

* Every word needs to be broken down into a tuple of characters
* Frequency is counted
* Example:
```js
"low" (2 times) → {('l', 'o', 'w'): 2}
"lower" (1 time) → {('l', 'o', 'w', 'e', 'r'): 1}
Combined: {('l', 'o', 'w'): 2, ('l', 'o', 'w', 'e', 'r'): 1}

```

Step 3: Count adjacent pairs

* In every word tuple, count the pairs of adjacent characters
* Example:
```js
('l', 'o', 'w') has pairs: ('l', 'o') and ('o', 'w')
('l', 'o', 'w', 'e', 'r') has pairs: ('l', 'o'), ('o', 'w'), ('w', 'e'), ('e', 'r')

Total counts:
('l', 'o'): 2 + 1 = 3
('o', 'w'): 2 + 1 = 3
('w', 'e'): 1
('e', 'r'): 1
```

Step 4: Find most frequent pair

* Find the pair that has the most frequency
* Example `('l', 'o')` and `('o', 'w')` both has a count of 3. Let's assume that we choose `('l', 'o')`

Step 5: Merge the pair

* Wherever this pair is present in the corpus merge it
* Example:
```js
('l', 'o', 'w') → ('lo', 'w')
('l', 'o', 'w', 'e', 'r') → ('lo', 'w', 'e', 'r')
```
* Add the new token to the vocab: `vocab.add('lo')`

Step 6: Repeat

* Step 3-5 is repeated until the `len(vocab) < target_vocab_size` condition is met

<b>Output:</b> Final vocabulary and merges list

In [13]:
# Implementing a naive word-level tokenizer
import re
from collections import Counter

def naive_tok(text):
    vocab = {}
    text = text.lower()
    count = 0
    tokens = text.split()

    for word in tokens:
        word = re.sub(r'[^\w\s]', '', word)

    for word in tokens:
        if word not in vocab.keys():
            vocab[word] = count
            count += 1
        else:
            count += 1

    return tokens, vocab

tokens, vocab = naive_tok("Hello world this is a test")
print(tokens, vocab, sep="\n")

['hello', 'world', 'this', 'is', 'a', 'test']
{'hello': 0, 'world': 1, 'this': 2, 'is': 3, 'a': 4, 'test': 5}


In [15]:
def train_bpe(corpus, vocab_size):
    for i in range(len(corpus)):
        corpus[i] = corpus[i].lower()

    corpus_dict = {
        tuple(word): freq
        for word, freq in Counter(corpus).items()
    }

    vocab = set(char for word in corpus_dict.keys() for char in word)
    merges = []

    while len(vocab) < vocab_size:
        
        pair_counts = {}
        
        for char_tuple, freq in corpus_dict.items():
            for i in range(len(char_tuple) - 1):
                pair = (char_tuple[i], char_tuple[i+1])
                
                if pair in pair_counts:
                    pair_counts[pair] += freq
                else:
                    pair_counts[pair] = freq
        
        if not pair_counts:
            break

        best_pair = max(pair_counts, key=pair_counts.get)
        token = best_pair[0] + best_pair[1]
        merges.append(best_pair)
        
        new_corpus_dict = {}
        
        for word, freq in corpus_dict.items():
            word_list = list(word)  
            i = 0
            
            while i < len(word_list) - 1:
                if word_list[i] + word_list[i+1] == token:
                    word_list[i:i+2] = [token]  
                else:
                    i += 1
            
            new_corpus_dict[tuple(word_list)] = freq
        
        corpus_dict = new_corpus_dict
        vocab.add(token)
        
        print(f"Merged: {token} | Vocab size: {len(vocab)}")
    
    return vocab, merges

train_bpe(["low", "low", "lower"], 10)

Merged: lo | Vocab size: 6
Merged: low | Vocab size: 7
Merged: lowe | Vocab size: 8
Merged: lower | Vocab size: 9


({'e', 'l', 'lo', 'low', 'lowe', 'lower', 'o', 'r', 'w'},
 [('l', 'o'), ('lo', 'w'), ('low', 'e'), ('lowe', 'r')])